## Test-set reconstruction + KL evaluator (Kaggle)

This notebook evaluates trained checkpoints on the **test memmap split**.

For each model checkpoint:
- **Reconstruction loss** (token CE with PAD ignored)
- **KL bits/dim** (for VAE models only)

### Evaluation protocol
- Sample **1,000 random batches** from the test split (with replacement).
- Compute mean recon loss and mean KL bits/dim across those batches.

### Notes
- Most models use `block_size=1024`.
- `VAE v4` and `Plain v4` use `block_size=650` (we truncate test sequences to 650 for those models).


In [41]:
from __future__ import annotations

import json
import math
import os
import time
from contextlib import nullcontext
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Optional

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset, RandomSampler
from tqdm.auto import tqdm

print('cwd:', os.getcwd())
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('torch cuda:', torch.version.cuda)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

IS_KAGGLE = Path('/kaggle').exists() and Path('/kaggle/input').exists()

INPUT_ROOT = Path('/kaggle/input') if IS_KAGGLE else Path.cwd()
WORKING_ROOT = Path('/kaggle/working') if IS_KAGGLE else (Path.cwd() / 'artifacts' / 'local_eval')
WORKING_ROOT.mkdir(parents=True, exist_ok=True)

print('IS_KAGGLE:', IS_KAGGLE)
print('INPUT_ROOT:', INPUT_ROOT)
print('WORKING_ROOT:', WORKING_ROOT)

if IS_KAGGLE:
    print('--- /kaggle/input ---')
    for p in sorted(INPUT_ROOT.iterdir()):
        if p.is_dir():
            print(' -', p.name)


cwd: /Users/kshoker/Desktop/CPSC 440/project/kaggle/notebooks
torch: 2.10.0
cuda available: False
torch cuda: None
IS_KAGGLE: False
INPUT_ROOT: /Users/kshoker/Desktop/CPSC 440/project/kaggle/notebooks
WORKING_ROOT: /Users/kshoker/Desktop/CPSC 440/project/kaggle/notebooks/artifacts/local_eval


In [42]:
# -------------------------
# Config (edit these on Kaggle)
# -------------------------

KAGGLE_USER = "karandeepshoker"

# Test memmap dataset slug (Kaggle)
TEST_DATASET = "preprocessed-memmaptest"

# Dataset folder under /kaggle/input that contains your checkpoints (Kaggle only).
MODELS_DATASET = "<set-me>"  # set on Kaggle

# Local paths (used when not on Kaggle)
LOCAL_TEST_SPLIT_DIR = os.environ.get("LOCAL_TEST_SPLIT_DIR", "")
# Optional: alternate test split for models trained on the "small" preprocessing
LOCAL_TEST_SPLIT_DIR_SMALL = os.environ.get("LOCAL_TEST_SPLIT_DIR_SMALL", "")

# Inside each dataset, where do the split folders live?
SPLIT_ROOT_REL = Path('.')

# Evaluation
# QUICK_MODE is the ~"1 minute per model" rough estimate.
# FAST_MODE is a still-reasonable but more stable estimate.
QUICK_MODE = True
FAST_MODE = not QUICK_MODE

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if (DEVICE == 'cuda') else torch.float32
USE_AMP = (DEVICE == 'cuda')

# Keep the code-path identical; only reduce sampling.
if QUICK_MODE:
    EVAL_BATCHES = 10
    EVAL_BATCH_SIZE = 16 if DEVICE == 'cuda' else 4
elif FAST_MODE:
    EVAL_BATCHES = 100
    EVAL_BATCH_SIZE = 16
else:
    EVAL_BATCHES = 1_000
    EVAL_BATCH_SIZE = 16

# IMPORTANT (local/Jupyter): multiprocessing DataLoader workers can't pickle notebook-defined classes.
# Use 0 workers locally; keep >0 on Kaggle.
NUM_WORKERS = 2 if IS_KAGGLE else 0
PIN_MEMORY = True if (IS_KAGGLE and DEVICE == 'cuda') else False
PERSISTENT_WORKERS = True if (IS_KAGGLE and NUM_WORKERS > 0) else False

try:
    torch.set_float32_matmul_precision('high')
except Exception:
    pass

print('DEVICE:', DEVICE, 'DTYPE:', DTYPE, 'USE_AMP:', USE_AMP)
print('QUICK_MODE:', QUICK_MODE, 'FAST_MODE:', FAST_MODE)
print('EVAL_BATCHES:', EVAL_BATCHES, 'EVAL_BATCH_SIZE:', EVAL_BATCH_SIZE)

if IS_KAGGLE:
    TEST_SPLIT_DIR = INPUT_ROOT / 'datasets' / KAGGLE_USER / TEST_DATASET / SPLIT_ROOT_REL / 'test'
    MODELS_ROOT = INPUT_ROOT / MODELS_DATASET
    print('TEST_SPLIT_DIR:', TEST_SPLIT_DIR)
    print('MODELS_ROOT:', MODELS_ROOT)

    if MODELS_DATASET == "<set-me>":
        raise ValueError("Set MODELS_DATASET to the folder under /kaggle/input that contains your checkpoints")
    if not MODELS_ROOT.exists():
        raise FileNotFoundError(f"MODELS_ROOT does not exist: {MODELS_ROOT}")
    if not TEST_SPLIT_DIR.exists():
        raise FileNotFoundError(f"TEST_SPLIT_DIR does not exist: {TEST_SPLIT_DIR}")
else:
    # Local: find repo root by searching upwards for ml/src/musicgen/runs
    here = Path.cwd().resolve()
    repo_root = None
    for p in [here] + list(here.parents)[:10]:
        if (p / "ml" / "src" / "musicgen" / "runs").is_dir():
            repo_root = p
            break
    if repo_root is None:
        raise FileNotFoundError("Could not locate repo root containing ml/src/musicgen/runs")

    MODELS_ROOT = repo_root

    # If env var not set, try to find artifacts/preprocessed_memmap/test by searching upwards.
    if not LOCAL_TEST_SPLIT_DIR:
        for p in [here] + list(here.parents)[:10]:
            candidate = p / "artifacts" / "preprocessed_memmap" / "test"
            if candidate.exists():
                LOCAL_TEST_SPLIT_DIR = str(candidate)
                break

    # If env var not set, try to find artifacts/preprocessed_memmap_small/test by searching upwards.
    if not LOCAL_TEST_SPLIT_DIR_SMALL:
        for p in [here] + list(here.parents)[:10]:
            candidate = p / "artifacts" / "preprocessed_memmap_small" / "test"
            if candidate.exists():
                LOCAL_TEST_SPLIT_DIR_SMALL = str(candidate)
                break

    if not LOCAL_TEST_SPLIT_DIR:
        raise ValueError(
            "Set LOCAL_TEST_SPLIT_DIR env var (or ensure artifacts/preprocessed_memmap/test exists) containing: "
            "meta.json, X.u16.memmap, bar_indices.u8.memmap, attributes.u8.memmap"
        )

    TEST_SPLIT_DIR = Path(LOCAL_TEST_SPLIT_DIR).expanduser().resolve()

    print('TEST_SPLIT_DIR:', TEST_SPLIT_DIR)
    print('MODELS_ROOT:', MODELS_ROOT)

    if not TEST_SPLIT_DIR.exists():
        raise FileNotFoundError(f"TEST_SPLIT_DIR does not exist: {TEST_SPLIT_DIR}")


DEVICE: cpu DTYPE: torch.float32 USE_AMP: False
QUICK_MODE: True FAST_MODE: False
EVAL_BATCHES: 10 EVAL_BATCH_SIZE: 4
TEST_SPLIT_DIR: /Users/kshoker/Desktop/CPSC 440/project/artifacts/preprocessed_memmap/test
MODELS_ROOT: /Users/kshoker/Desktop/CPSC 440/project


In [43]:
def require_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")


def load_meta(split_dir: Path) -> dict[str, Any]:
    meta_path = split_dir / 'meta.json'
    require_file(meta_path)
    with meta_path.open('r', encoding='utf-8') as f:
        return json.load(f)


def _meta_int(meta: dict[str, Any], keys: list[str], *, default: int | None = None) -> int:
    for k in keys:
        v = meta.get(k)
        if v is None:
            continue
        try:
            return int(v)
        except Exception:
            pass
    if default is None:
        raise KeyError(f"Missing required meta keys (tried {keys})")
    return int(default)


class MemmapMusicDataset(Dataset):
    def __init__(self, split_dir: Path):
        self.split_dir = split_dir
        require_file(split_dir / 'X.u16.memmap')
        require_file(split_dir / 'bar_indices.u8.memmap')
        require_file(split_dir / 'attributes.u8.memmap')

        self.meta = load_meta(split_dir)

        token_ids = self.meta.get('token_ids', {})
        if isinstance(token_ids, dict) and 'pad_id' in token_ids:
            self.pad_id = int(token_ids['pad_id'])
        else:
            # legacy meta layout
            self.pad_id = _meta_int(self.meta, ['pad_id'], default=0)

        self.n = _meta_int(self.meta, ['N', 'num_examples'])
        self.t = _meta_int(self.meta, ['T', 'tokens_per_example'])
        self.bars = _meta_int(self.meta, ['bars', 'bars_per_sample'], default=8)
        self.num_attributes = _meta_int(self.meta, ['num_attributes'], default=4)

        self._x = np.memmap(split_dir / 'X.u16.memmap', mode='r', dtype=np.uint16, shape=(self.n, self.t))
        self._bi = np.memmap(split_dir / 'bar_indices.u8.memmap', mode='r', dtype=np.uint8, shape=(self.n, self.t))
        self._a = np.memmap(split_dir / 'attributes.u8.memmap', mode='r', dtype=np.uint8, shape=(self.n, self.bars, self.num_attributes))

    def __len__(self) -> int:
        return self.n

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        X = torch.from_numpy(self._x[idx].astype(np.int64, copy=False))
        bar_indices = torch.from_numpy(self._bi[idx].astype(np.int64, copy=False))
        attributes = torch.from_numpy(self._a[idx].astype(np.int64, copy=False))

        # Y = shift(X) with pad appended (per your spec)
        pad = int(self.pad_id)
        Y = torch.empty_like(X)
        Y[:-1] = X[1:]
        Y[-1] = pad

        return {'X': X, 'Y': Y, 'bar_indices': bar_indices, 'attributes': attributes}


def _resolve_split_dir_for_spec(spec: Optional['ModelSpec']) -> Path:
    if spec is None or not spec.test_split_relpath:
        return TEST_SPLIT_DIR

    if IS_KAGGLE:
        # On Kaggle, this expects your input dataset contains that relative path.
        return (INPUT_ROOT / MODELS_DATASET / spec.test_split_relpath).resolve()

    # Local: interpret as repo-root-relative
    return (MODELS_ROOT / spec.test_split_relpath).resolve()


def get_dataset_for_spec(spec: Optional['ModelSpec'] = None) -> MemmapMusicDataset:
    split_dir = _resolve_split_dir_for_spec(spec)
    return MemmapMusicDataset(split_dir)


def _describe_ds(ds: MemmapMusicDataset, *, label: str):
    print(label, 'dir:', ds.split_dir)
    print(label, 'N:', len(ds), 'T:', ds.t, 'pad_id:', ds.pad_id)


# Default dataset (used for most models)
test_ds = get_dataset_for_spec(None)
_describe_ds(test_ds, label='test')


test dir: /Users/kshoker/Desktop/CPSC 440/project/artifacts/preprocessed_memmap/test
test N: 21105 T: 1024 pad_id: 0


In [44]:
_loader_cache: dict[str, tuple[MemmapMusicDataset, DataLoader]] = {}


def get_loader_for_spec(spec: Optional['ModelSpec'] = None) -> tuple[MemmapMusicDataset, DataLoader]:
    ds = get_dataset_for_spec(spec)
    key = str(ds.split_dir)
    if key in _loader_cache:
        return _loader_cache[key]

    # DataLoader: random batches with replacement for i.i.d-ish sampling
    sampler = RandomSampler(ds, replacement=True, num_samples=EVAL_BATCHES * EVAL_BATCH_SIZE)

    loader = DataLoader(
        ds,
        batch_size=EVAL_BATCH_SIZE,
        sampler=sampler,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
    )

    _loader_cache[key] = (ds, loader)
    print('loader ready for:', key)
    return ds, loader


# Default loader (used for most models)
test_ds, test_loader = get_loader_for_spec(None)


loader ready for: /Users/kshoker/Desktop/CPSC 440/project/artifacts/preprocessed_memmap/test


In [45]:
def load_checkpoint(path: Path) -> dict[str, Any]:
    require_file(path)
    ckpt = torch.load(path, map_location='cpu')
    if not isinstance(ckpt, dict):
        raise ValueError('Checkpoint must be a dict.')
    return ckpt


def extract_state_dict(ckpt: dict[str, Any]) -> dict[str, torch.Tensor]:
    state = ckpt.get('model', ckpt.get('model_state_dict', ckpt.get('state_dict')))
    if state is None:
        # some saves are raw state dict
        if all(isinstance(k, str) for k in ckpt.keys()):
            return ckpt  # type: ignore[return-value]
        raise ValueError('Could not find model state dict in checkpoint.')
    if not isinstance(state, dict):
        raise ValueError('Model state dict must be a dict.')
    return state


def kl_bits_per_dim_from_out(out: dict[str, Any]) -> Optional[float]:
    k = out.get('kl_per_dim_mean')
    if k is None:
        return None
    if not torch.is_tensor(k):
        return None
    # k is in nats; convert to bits and average across dims
    return float((k.mean() / math.log(2.0)).detach().cpu().item())


@dataclass(frozen=True)
class ModelSpec:
    name: str
    arch: str  # 'plain' | 'vae' | 'simple_vae'
    ckpt_relpath: str
    block_size: int
    # Optional override to evaluate against a different test split.
    # Examples (local): "artifacts/preprocessed_memmap_small/test"
    test_split_relpath: Optional[str] = None


# Checkpoint paths
# - Local: uses repo layout under ml/src/musicgen/runs/*/ckpt.pt
# - Kaggle: works if your checkpoint dataset contains the same repo-relative paths
ALL_MODEL_SPECS: list[ModelSpec] = [
    ModelSpec('VAE v1', 'vae', 'ml/src/musicgen/runs/vae_v1/ckpt.pt', 1024),
    ModelSpec('VAE v2', 'vae', 'ml/src/musicgen/runs/vae_v2/ckpt.pt', 1024),
    ModelSpec('VAE v3', 'vae', 'ml/src/musicgen/runs/vae_v3/ckpt.pt', 1024),
    ModelSpec('VAE v4', 'vae', 'ml/src/musicgen/runs/vae_v4/ckpt.pt', 650, test_split_relpath='artifacts/preprocessed_memmap_small/test'),

    ModelSpec('Simple v1', 'simple_vae', 'ml/src/musicgen/runs/simple_v1/ckpt.pt', 1024),
    # Simple v2 ckpt path TBD (folder exists locally but no ckpt.pt yet)
    # ModelSpec('Simple v2', 'simple_vae', 'ml/src/musicgen/runs/simple_v2/ckpt.pt', 1024),

    ModelSpec('Plain v1', 'plain', 'ml/src/musicgen/runs/plain_v1/ckpt.pt', 1024),
    ModelSpec('Plain v2', 'plain', 'ml/src/musicgen/runs/plain_v2/ckpt.pt', 1024),
    ModelSpec('Plain v3', 'plain', 'ml/src/musicgen/runs/plain_v3/ckpt.pt', 1024),
    ModelSpec('Plain v4', 'plain', 'ml/src/musicgen/runs/plain_v4/ckpt.pt', 650, test_split_relpath='artifacts/preprocessed_memmap_small/test'),
]

# In FAST_MODE, evaluate a smaller representative set first.
MODEL_SPECS = (
    [
        next(s for s in ALL_MODEL_SPECS if s.name == 'VAE v4'),
        next(s for s in ALL_MODEL_SPECS if s.name == 'VAE v3'),
        next(s for s in ALL_MODEL_SPECS if s.name == 'Simple v1'),
        next(s for s in ALL_MODEL_SPECS if s.name == 'Plain v4'),
        next(s for s in ALL_MODEL_SPECS if s.name == 'Plain v3'),
    ]
    if FAST_MODE
    else ALL_MODEL_SPECS
)

print('FAST_MODE:', FAST_MODE)
print('models:', len(MODEL_SPECS), 'of', len(ALL_MODEL_SPECS))


FAST_MODE: False
models: 9 of 9


In [46]:
def build_model(spec: ModelSpec, ckpt: dict[str, Any]):
    cfg_dict = ckpt.get('cfg', {}) if isinstance(ckpt, dict) else {}

    if spec.arch == 'plain':
        from musicgen.models.plain_transformer import PlainTransformerConfig, PlainTransformerDecoder

        cfg = PlainTransformerConfig(**cfg_dict) if isinstance(cfg_dict, dict) and cfg_dict else PlainTransformerConfig()
        cfg = PlainTransformerConfig(**{**cfg.__dict__, 'block_size': int(spec.block_size)})
        model = PlainTransformerDecoder(cfg)
        return model

    if spec.arch == 'vae':
        from musicgen.models.vae import MusicVAE, VAEConfig

        cfg = VAEConfig(**cfg_dict) if isinstance(cfg_dict, dict) and cfg_dict else VAEConfig()
        cfg = VAEConfig(**{**cfg.__dict__, 'block_size': int(spec.block_size)})
        model = MusicVAE(cfg)
        return model

    if spec.arch == 'simple_vae':
        from musicgen.models.simple_vae import SimpleMusicVAE, SimpleVAEConfig

        cfg = SimpleVAEConfig(**cfg_dict) if isinstance(cfg_dict, dict) and cfg_dict else SimpleVAEConfig()
        cfg = SimpleVAEConfig(**{**cfg.__dict__, 'block_size': int(spec.block_size)})
        model = SimpleMusicVAE(cfg)
        return model

    raise ValueError(f'Unknown arch: {spec.arch}')


def move_batch(batch: dict[str, torch.Tensor], *, block_size: int) -> dict[str, torch.Tensor]:
    X = batch['X']
    Y = batch['Y']
    bar_indices = batch['bar_indices']
    attributes = batch['attributes']

    if block_size != X.shape[1]:
        X = X[:, :block_size]
        Y = Y[:, :block_size]
        bar_indices = bar_indices[:, :block_size]

        # Slicing produces non-contiguous views; some model code uses .view(...)
        X = X.contiguous()
        Y = Y.contiguous()
        bar_indices = bar_indices.contiguous()

    return {
        'X': X.to(device=DEVICE, non_blocking=True),
        'Y': Y.to(device=DEVICE, non_blocking=True),
        'bar_indices': bar_indices.to(device=DEVICE, non_blocking=True),
        'attributes': attributes.to(device=DEVICE, non_blocking=True),
    }


@torch.no_grad()
def eval_model(spec: ModelSpec) -> dict[str, Any]:
    ckpt_path = MODELS_ROOT / spec.ckpt_relpath
    ckpt = load_checkpoint(ckpt_path)
    state = extract_state_dict(ckpt)

    model = build_model(spec, ckpt)
    model.load_state_dict(state, strict=True)
    model.to(device=DEVICE)
    if DEVICE == 'cuda' and DTYPE == torch.float16:
        model.to(dtype=torch.float16)
    model.eval()

    ds, loader = get_loader_for_spec(spec)
    pad_id = int(ds.pad_id)

    t0 = time.time()
    recon_losses: list[float] = []
    kl_bits: list[float] = []

    it = iter(loader)
    pbar = tqdm(range(EVAL_BATCHES), desc=f"{spec.name}", leave=False)
    for i in pbar:
        batch = next(it)
        b = move_batch(batch, block_size=int(spec.block_size))

        ctx = (
            torch.autocast(device_type='cuda', dtype=torch.float16)
            if (USE_AMP and DEVICE == 'cuda')
            else nullcontext()
        )
        with ctx:
            if spec.arch == 'plain':
                out = model(b['X'], b['bar_indices'], b['attributes'], targets=b['Y'], pad_id=pad_id)
                recon_losses.append(float(out['loss'].detach().cpu().item()))
            else:
                out = model(b['X'], b['bar_indices'], b['attributes'], targets=b['Y'], pad_id=pad_id, beta=1.0)
                recon_losses.append(float(out['loss_recon'].detach().cpu().item()))
                kb = kl_bits_per_dim_from_out(out)
                if kb is not None:
                    kl_bits.append(kb)

        if (i + 1) % 5 == 0:
            pbar.set_postfix(recon=float(np.mean(recon_losses)))

    res: dict[str, Any] = {
        'model': spec.name,
        'arch': spec.arch,
        'block_size': spec.block_size,
        'ckpt': str(ckpt_path),
        'eval_batches': EVAL_BATCHES,
        'eval_batch_size': EVAL_BATCH_SIZE,
        'recon_mean': float(np.mean(recon_losses)),
        'recon_std': float(np.std(recon_losses)),
        'kl_bits_dim_mean': float(np.mean(kl_bits)) if kl_bits else None,
        'kl_bits_dim_std': float(np.std(kl_bits)) if kl_bits else None,
        'seconds': float(time.time() - t0),
    }
    return res


In [47]:
# Ensure `musicgen` is importable
import sys

if IS_KAGGLE:
    # If you attached the repo as a dataset, set REPO_DATASET and uncomment.
    # REPO_DATASET = "<your-repo-dataset-slug>"
    # sys.path.insert(0, str(INPUT_ROOT / REPO_DATASET / 'ml' / 'src'))
    pass
else:
    # Use the repo root we detected in the config cell.
    sys.path.insert(0, str(MODELS_ROOT / "ml" / "src"))

import musicgen  # noqa: F401
print('musicgen import ok')


musicgen import ok


In [48]:
results = []

for spec in MODEL_SPECS:
    print('\n===', spec.name, '===')
    try:
        r = eval_model(spec)
        results.append(r)
        print('recon mean:', r['recon_mean'])
        print('kl bits/dim mean:', r['kl_bits_dim_mean'])
    except Exception as e:
        results.append({'model': spec.name, 'arch': spec.arch, 'error': repr(e)})
        print('ERROR:', repr(e))

out_path = WORKING_ROOT / 'test_recon_kl_results.json'
out_path.write_text(json.dumps(results, indent=2), encoding='utf-8')
print('wrote:', out_path)

# Display compact summary
for r in results:
    if 'error' in r:
        print(f"{r['model']}: ERROR")
    else:
        kb = r['kl_bits_dim_mean']
        kb_s = '—' if kb is None else f"{kb:.3f}"
        print(f"{r['model']}: recon={r['recon_mean']:.4f}  kl_bits/dim={kb_s}")



=== VAE v1 ===


VAE v1:   0%|          | 0/10 [00:00<?, ?it/s]

recon mean: 1.9004732728004456
kl bits/dim mean: 0.7244411587715149

=== VAE v2 ===


VAE v2:   0%|          | 0/10 [00:00<?, ?it/s]

recon mean: 1.7375375151634216
kl bits/dim mean: 0.7297065556049347

=== VAE v3 ===


...

recon mean: 1.7671534061431884
kl bits/dim mean: None

=== Plain v3 ===


Plain v3:   0%|          | 0/10 [00:00<?, ?it/s]

recon mean: 1.7740022778511046
kl bits/dim mean: None

=== Plain v4 ===


Plain v4:   0%|          | 0/10 [00:00<?, ?it/s]

recon mean: 1.713206160068512
kl bits/dim mean: None
wrote: /Users/kshoker/Desktop/CPSC 440/project/kaggle/notebooks/artifacts/local_eval/test_recon_kl_results.json
VAE v1: recon=1.9005  kl_bits/dim=0.724
VAE v2: recon=1.7375  kl_bits/dim=0.730
VAE v3: recon=1.6921  kl_bits/dim=0.859
VAE v4: recon=1.7385  kl_bits/dim=0.916
Simple v1: recon=1.9391  kl_bits/dim=0.800
Plain v1: recon=1.9447  kl_bits/dim=—
Plain v2: recon=1.7672  kl_bits/dim=—
Plain v3: recon=1.7740  kl_bits/dim=—
Plain v4: recon=1.7132  kl_bits/dim=—
